In [42]:
import pandas as pd
import geopandas as gpd

# --- Load Crimes CSV ---
df = pd.read_csv("Crimes_-_2001_to_Present_20251023.csv")

# Keep only relevant columns
df = df[['Case Number', 'Primary Type', 'Location Description', 'Latitude', 'Longitude', 'IUCR', 'Arrest', 'Domestic']].dropna(subset=['Latitude', 'Longitude'])

# --- Convert to GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs="EPSG:4326"  # WGS84 coordinate system
)

# --- Load Community Areas GeoJSON ---
communities = gpd.read_file("Boundaries_-_Community_Areas_20251023.geojson").to_crs("EPSG:4326")

# --- Spatial Join: assign each crime to its community ---
gdf_joined = gpd.sjoin(gdf, communities, how="inner", predicate="within")

# --- Clean columns ---
gdf_joined = gdf_joined.rename(columns={'community': 'Community Name', 'area_numbe': 'Community ID'})
gdf_joined = gdf_joined.drop(columns=['index_right', ':id', ':version', ':created_at', ':updated_at', 'shape_area', 'shape_len'], errors='ignore')

# --- Check results ---
print(gdf_joined[['Community ID', 'Community Name']].head())

print(f"\n✅ Spatial join complete! {len(gdf_joined)} crimes successfully matched to communities.")


  Community ID  Community Name
0           59   MCKINLEY PARK
1           67  WEST ENGLEWOOD
2           24       WEST TOWN
3           43     SOUTH SHORE
4           70         ASHBURN

✅ Spatial join complete! 1348011 crimes successfully matched to communities.


In [43]:
import plotly.express as px

# --- Dataset base: gdf_joined ---
crime_counts = (
    gdf_joined
    .groupby(['Primary Type', 'Location Description'])
    .size()
    .reset_index(name='Count')
)

# --- Top 10 Primary Types by total crimes ---
top_types = (
    crime_counts.groupby('Primary Type')['Count']
    .sum()
    .nlargest(10)
    .index
)
crime_counts_top = crime_counts[crime_counts['Primary Type'].isin(top_types)]

# --- Total by type (para color) ---
crime_counts_top['Total by Type'] = (
    crime_counts_top.groupby('Primary Type')['Count'].transform('sum')
)

# --- Treemap interactivo ---
fig1 = px.treemap(
    crime_counts_top,
    path=['Primary Type', 'Location Description'],
    values='Count',
    color='Total by Type',
    color_continuous_scale='Viridis',
    title='Treemap of Crime Frequency by Type and Location (Top 10, Spatial Join)',
)

fig1.update_traces(
    textinfo='label+value+percent parent',
    hovertemplate='<b>%{label}</b><br>Crimes: %{value}<br>Share: %{percentParent:.2%}<extra></extra>'
)

fig1.show()


/tmp/ipython-input-2556646637.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [47]:
# --- Clasificación de Severidad ---
violent_crimes = [
    'HOMICIDE', 'ASSAULT', 'BATTERY', 'ROBBERY',
    'CRIM SEXUAL ASSAULT', 'ARSON', 'KIDNAPPING'
]

gdf_joined['Severity'] = gdf_joined['Primary Type'].apply(
    lambda x: 'Violent' if x in violent_crimes else 'Non-Violent'
)

# --- Agrupación por Comunidad y Severidad ---
crime_summary = (
    gdf_joined.groupby(['Community Name', 'Severity'], as_index=False)
    .agg(total_cases=('Case Number', 'count'),
         arrest_rate=('Arrest', lambda x: x.mean()))
)

# --- Treemap Interactivo ---
fig2 = px.treemap(
    crime_summary,
    path=['Community Name', 'Severity'],
    values='total_cases',
    color='arrest_rate',
    color_continuous_scale='Viridis',
    title='Crimes by Community and Severity (Spatial Join)',
    hover_data={'arrest_rate': ':.2f', 'total_cases': True}
)

fig2.update_layout(
    title_font_size=20,
    margin=dict(t=60, l=25, r=25, b=25),
    uniformtext=dict(minsize=12, mode='hide')
)

fig2.show()


In [49]:
print(gdf_joined.columns)


Index(['Case Number', 'Primary Type', 'Location Description', 'Latitude',
       'Longitude', 'IUCR', 'Arrest', 'Domestic', 'geometry', 'Community ID',
       'Community Name', 'area_num_1', 'Severity'],
      dtype='object')


In [50]:
# ============================================
# 🚨 TREEMAP (TOP 10): Crimes by Community and Severity (Spatial Join)
# ============================================

import plotly.express as px
import pandas as pd

# --- Create severity classification ---
violent_crimes = [
    'HOMICIDE', 'ASSAULT', 'BATTERY', 'ROBBERY',
    'CRIM SEXUAL ASSAULT', 'ARSON', 'KIDNAPPING'
]

gdf_joined['Severity'] = gdf_joined['Primary Type'].apply(
    lambda x: 'Violent' if x in violent_crimes else 'Non-Violent'
)

# --- Group by community and severity ---
crime_summary = (
    gdf_joined.groupby(['Community Name', 'Severity'], as_index=False)
    .agg(
        total_cases=('Case Number', 'count'),
        arrest_rate=('Arrest', lambda x: x.mean())
    )
)

# --- Clean missing data ---
crime_summary['arrest_rate'] = crime_summary['arrest_rate'].fillna(0)
crime_summary['Community Name'] = crime_summary['Community Name'].fillna('Unknown')

# --- Select TOP 10 communities by total cases ---
top_10_communities = (
    crime_summary.groupby('Community Name')['total_cases']
    .sum()
    .nlargest(10)
    .index
)

crime_top10 = crime_summary[crime_summary['Community Name'].isin(top_10_communities)]

# --- Create Treemap ---
fig = px.treemap(
    crime_top10,
    path=['Community Name', 'Severity'],
    values='total_cases',
    color='arrest_rate',
    color_continuous_scale='Viridis',
    title='Top 10 Communities by Crime Volume and Severity (Spatial Join)',
    hover_data={'arrest_rate': ':.2f', 'total_cases': True}
)

fig.update_traces(
    textinfo='label+value+percent parent',
    hovertemplate='<b>%{label}</b><br>Crimes: %{value}<br>Share: %{percentParent:.2%}<extra></extra>'
)

fig.update_layout(
    title_font_size=22,
    margin=dict(t=60, l=25, r=25, b=25),
    uniformtext=dict(minsize=12, mode='show')
)

fig.show()
